### Ingestion del archivo "movie_languages.json"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

#### Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
movies_languages_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True),
    StructField("languageId", IntegerType(), True),
    StructField("languageRoleId", IntegerType(), True)
    ])

In [0]:
movies_languages_df = spark.read \
            .schema(movies_languages_schema) \
            .option("multiLine", "true") \
            .json(f"{bronze_folder_path}/{v_file_date}/movie_language")

# movie_languages_*.csv

In [0]:
display(movies_languages_df)

In [0]:
movies_languages_df.count()   # ver la cantidad de registros que se han cargado 

#### Paso 2 - renombrar columnas y agregar columnas nuevas
1. renombrar "movieId" a "movie_id"
2. renombrar "languageId" a "language_id"
3. Agregar las columnas "ingestion_date" y "environment"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movies_languages_withcolumns_df = add_ingestion_date(movies_languages_df) \
                            .withColumnsRenamed({"movieId": "movie_id", "languageId": "language_id"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))

In [0]:
display(movies_languages_withcolumns_df)

#### Paso 3 - Eliminar "languageRoleId"

In [0]:
from pyspark.sql.functions import col

In [0]:
movies_languages_final_df = movies_languages_withcolumns_df.drop(col("languageRoleId"))

#### Paso 4 - Escribir salida en un formato "Parquet"

In [0]:
# movies_languages_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/movies_languages")

In [0]:
# display(spark.read.parquet("abfss://silver@moviehistory22.dfs.core.windows.net/movies_languages"))

In [0]:
# overwrite_partition("movie_silver", "movies_languages","file_date",v_file_date)

In [0]:
# movies_languages_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies_languages")
# movies_languages_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies_languages")

In [0]:
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.language_id = src.language_id AND tgt.file_date=src.file_date'
merge_delta_lake(movies_languages_final_df, "movie_silver", "movies_languages", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movies_languages
GROUP BY file_date

In [0]:
%sql
SELECT * FROM movie_silver.movies_languages

In [0]:
dbutils.notebook.exit("Success")